# Actividad 1 · Un potencial, dos parámetros
### Lennard-Jones: la base física del *scoring* en docking molecular
**Taller:** Física y matemática detrás de los campos de fuerza y la dinámica molecular · XXXIV Semana de la Química, Bogotá 2026

---

**Duración:** ~25 min · **Herramienta:** Google Colab o Jupyter · **Prerrequisitos:** ninguno

**Objetivos:**
1. Entender qué es un potencial de interacción entre dos átomos.
2. Derivar la fuerza a partir del potencial (regla de la cadena, nivel bachillerato).
3. Visualizar cómo los dos parámetros del potencial ($\varepsilon$ y $\sigma$) controlan la unión.
4. Conectar todo esto con el *score* que usa el software de docking.

## Concepto clave

Dos átomos interactúan mediante el potencial de **Lennard-Jones**:

$$U(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

- $r$: distancia entre los centros de los dos átomos.
- $\varepsilon$ (**epsilon**): profundidad del pozo → *qué tan fuerte se unen*.
- $\sigma$ (**sigma**): distancia donde el potencial se anula → *tamaño efectivo* del átomo.

> 🔑 **Idea central:** con solo dos parámetros ya se describe la "química" de un par de átomos. El término de van der Waals que usa el *scoring* del docking **es este potencial**. Entenderlo te permite *auditar* el score en lugar de aceptarlo como caja negra.

## Paso 0 · Importar librerías

Ejecuta la celda (▶). Solo necesitamos dos herramientas:
- `numpy`: para hacer matemática con listas de números (arrays).
- `matplotlib`: para graficar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Paso 1 · Los parámetros y la distancia

Trabajaremos en **unidades reducidas** (lo habitual en simulación): $\varepsilon = 1$ y $\sigma = 1$. Así los números son manejables y todo se puede reescalar después.

Creamos una lista de distancias $r$ desde $0.9\,\sigma$ hasta $3\,\sigma$. Empezamos en 0.9 (no en 0) porque cerca de $r=0$ el potencial crece enormemente y estropearía la escala de la gráfica — piensa por qué.

In [ ]:
# Parámetros del potencial (unidades reducidas)
epsilon = 1.0   # profundidad del pozo
sigma   = 1.0   # tamaño efectivo del átomo

# Distancias a evaluar (en unidades de sigma)
r = np.linspace(0.9, 3.0, 400)
print("Evaluaremos U(r) en", len(r), "distancias, desde r =", r[0], "hasta r =", round(r[-1],2))

## Paso 2 · El potencial U(r) — *COMPLETA AQUÍ*

Implementa el potencial de Lennard-Jones. En Python:

| Matemáticas | Python |
|---|---|
| $x^{12}$ | `x**12` |
| $\dfrac{\sigma}{r}$ | `sigma/r` |

$$U(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

In [ ]:
def U_lennard_jones(r, epsilon, sigma):
    """Potencial de Lennard-Jones U(r)."""
    # COMPLETA AQUÍ: return 4*epsilon*( ... )
    return 4 * epsilon * ((sigma / r)**12 - (sigma / r)**6)

# Probemos la función en dos distancias de ejemplo
print("U(1.0) =", round(U_lennard_jones(1.0, epsilon, sigma), 4), "  <- potencial se anula en r = sigma")
print("U(1.5) =", round(U_lennard_jones(1.5, epsilon, sigma), 4), "  <- ¿positivo o negativo? ¿qué significa cada caso?")

## Paso 3 · La fuerza F(r) — *DERIVA Y COMPLETA*

### ✏️ Primero, en papel (5 min)
La fuerza es el **gradiente negativo** del potencial (una dimensión):

$$F(r) = -\frac{dU}{dr}$$

Deriva $U(r)$ usando la regla de la cadena. Sugerencia: escribe $U(r) = 4\varepsilon\left[\sigma^{12}r^{-12} - \sigma^{6}r^{-6}\right]$ y deriva término a término. Debes llegar a:

$$F(r) = \frac{24\,\varepsilon}{r}\left[2\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

> 🤔 Antes de programar: ¿en qué valor de $r$ se anula $F$? ¿Qué significa físicamente ese punto?

### 💻 Ahora en código

Implemente la fuerza que derivado. Puede comprobarla numéricamente: la función `check` compara su $F(r)$ contra una derivada calculada por computadora (diferencias finitas).

In [ ]:
def F_lennard_jones(r, epsilon, sigma):
    """Fuerza sobre el átomo: F(r) = -dU/dr (¡la que derivaste a mano!)."""
    # COMPLETA AQUÍ: return (24*epsilon/r)*( ... )
    return (24 * epsilon / r) * (2 * (sigma / r)**12 - (sigma / r)**6)

# ---- Comprobación automática (no modificar) ----
def dU_numerica(r, epsilon, sigma, h=1e-5):
    """Derivada de U por diferencias finitas (la computadora deriva por nosotros)."""
    return (U_lennard_jones(r + h, epsilon, sigma) - U_lennard_jones(r - h, epsilon, sigma)) / (2 * h)

r_test = np.linspace(0.95, 2.5, 50)
error = np.max(np.abs(F_lennard_jones(r_test, epsilon, sigma) - (-dU_numerica(r_test, epsilon, sigma))))
print("Error máximo frente a la derivada numérica:", f"{error:.2e}")
print("✅ ¡Tu derivada analítica es correcta!" if error < 1e-6 else "⚠️ Revisa la fórmula de F(r).")

## Paso 4 · Graficar el potencial y la fuerza

Ejecuta la celda. Observa:

- **Gráfica superior (energía):** el famoso "pozo". El fondo del pozo está en $U = -\varepsilon$ y ocurre en $r = 2^{1/6}\sigma \approx 1.12\,\sigma$ → **esa es la distancia de equilibrio**.
- **Gráfica inferior (fuerza):** $F = 0$ exactamente donde $U$ tiene su mínimo (no es coincidencia: ¿por qué?).
- Para $r < r_{eq}$: $F > 0$ (repulsión, empuja hacia afuera). Para $r > r_{eq}$: $F < 0$ (atracción, jala hacia adentro). El sistema siempre "quiere" volver al fondo del pozo.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

# --- Energía ---
ax1.plot(r, U_lennard_jones(r, epsilon, sigma), color='darkred', lw=2)
ax1.axhline(0, color='gray', lw=0.8)
r_eq = 2**(1/6) * sigma
ax1.axvline(r_eq, color='gray', ls='--', lw=1)
ax1.annotate('distancia de equilibrio\n$U=-\\varepsilon$',
             xy=(r_eq, -epsilon), xytext=(1.7, -0.55),
             arrowprops=dict(arrowstyle='->', color='black'))
ax1.set_ylabel('U(r)  [energía]')
ax1.set_title('Potencial de Lennard-Jones: el "pozo" de unión')
ax1.grid(alpha=0.3)

# --- Fuerza ---
ax2.plot(r, F_lennard_jones(r, epsilon, sigma), color='navy', lw=2)
ax2.axhline(0, color='gray', lw=0.8)
ax2.axvline(r_eq, color='gray', ls='--', lw=1)
ax2.text(0.95, 30, 'repulsión ($F>0$)', color='navy')
ax2.text(2.2, -2.2, 'atracción ($F<0$)', color='navy')
ax2.set_xlabel('r  [distancia, en unidades de $\\sigma$]')
ax2.set_ylabel('F(r)  [fuerza]')
ax2.set_title('La fuerza siempre empuja hacia el equilibrio')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Paso 5 · Experimentos: ¿qué controla cada parámetro?

Es tu turno de **jugar con los parámetros**. Cambia los valores en la celda, ejecuta, y responde las preguntas de abajo.

| Experimento | Cambio | Pregunta |
|---|---|---|
| 1 | $\varepsilon = 2$ (doble) | ¿El pozo es más profundo o más superficial? ¿A qué distancia está el mínimo ahora? |
| 2 | $\sigma = 1.3$ | ¿Se mueve el mínimo? ¿En qué dirección? |
| 3 | $\varepsilon = 0.3$, $\sigma = 0.8$ | Predice *antes* de ejecutar: ¿pozo profundo o superficial? ¿cerca o lejos? |

In [ ]:
# ✏️ CAMBIA AQUÍ los valores y ejecuta de nuevo
epsilon_exp = 1.0
sigma_exp   = 1.0

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(r, U_lennard_jones(r, epsilon, sigma), color='gray', ls='--',
        label=f'referencia ($\\varepsilon$={epsilon}, $\\sigma$={sigma})')
ax.plot(r, U_lennard_jones(r, epsilon_exp, sigma_exp), color='darkred', lw=2,
        label=f'experimento ($\\varepsilon$={epsilon_exp}, $\\sigma$={sigma_exp})')
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('r  [unidades de $\\sigma$ de referencia]')
ax.set_ylabel('U(r)')
ax.set_title('Comparación de pozos')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

i = np.argmin(U_lennard_jones(r, epsilon_exp, sigma_exp))
print(f"Experimento: mínimo en r = {r[i]:.3f}  con  U = {U_lennard_jones(r[i], epsilon_exp, sigma_exp):.3f}")

## Cierre · Conexión con el *docking molecular*

1. **Relación con el score.** El *scoring* del docking ordena las poses sumando contribuciones:
$$\text{Score}_{\text{dock}} \approx E_{\text{vdW}} + E_{\text{electrostática}} + E_{\text{H-bond}} + \dots$$
   El término $E_{\text{vdW}}$ **es el potencial de Lennard-Jones** que acabas de programar, evaluado entre cada átomo del ligando y cada átomo de la proteína.

2. **Los parámetros en la práctica.** En los campos de fuerza reales, $\varepsilon$ y $\sigma$ dependen del tipo de átomo (C, O, N, H...): por eso la preparación correcta de ligandos y proteínas importa tanto.

3. **¿Dónde entra esto en el taller?** Es el primer escalón del mapa:
   - **Actividad 1 (esta):** el potencial → la fuerza. ⬅️ *estás aquí*
   - **Actividad 2:** bajar el paisaje energético → minimización (MMFF94) y optimización (búsqueda de poses).
   - **Actividad 3:** ensamblar la molécula completa → de Lennard-Jones a un campo de fuerza completo.
   - **Actividades 4–5:** mover los átomos en el tiempo → dinámica molecular y auditoría de hipótesis de unión.

> 💬 **Pregunta guía del curso:** *¿Cómo pasamos de una colección de productos naturales a hipótesis de unión molecular que sean reproducibles y honestas?* — Todo empieza por entender la función que puntúa las poses. Ya la conoces por dentro.